In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/qoute_dataset.csv")

In [ ]:
df.head()

In [ ]:
df["quote"][0]

In [ ]:
df.shape

In [ ]:
quotes = df["quote"]
quotes.head()

In [ ]:
quotes = quotes.str.lower()
quotes.head()

In [ ]:
import string
translator = str.maketrans("","" , string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))
quotes.head()

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab = 8978
token = Tokenizer(num_words = vocab )
token.fit_on_texts(quotes)

In [ ]:
word_index = token.word_index
print(len(word_index))
list(word_index.items())[:10]

In [ ]:
sequence = token.texts_to_sequences(quotes)
sequence[:5]

In [ ]:
for i in range(3):
  print(quotes[i])


In [ ]:
for i in range(3):
  print(sequence[i])

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Saare quotes ko ek text mein combine karo
text = " ".join(quotes.astype(str))

# WordCloud create karo
wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(text)

# Display
plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

In [ ]:
x = []
y = []

for seq in sequence:
  for i in range(1 , len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]

    x.append(input_seq)
    y.append(output_seq)

# print(x[:10])
# print(y[:10])

x

In [ ]:
y

In [ ]:
len(x)


In [ ]:
len(y)

In [ ]:
#For Same Size

max_len = max(len(x) for x in x)
print(max_len)

In [ ]:
from  tensorflow.keras.preprocessing.sequence import pad_sequences

padded_sequences = pad_sequences(
    x ,
    maxlen = max_len ,
    padding = "post",
    truncating = "post"

)

In [ ]:
padded_sequences

In [ ]:
y = np.array(y)

In [ ]:
padded_sequences.shape

In [ ]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y , num_classes = vocab)

In [ ]:
y.shape

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding , SimpleRNN , LSTM , Dense

In [ ]:
rnn_model = Sequential([
    Embedding(input_dim = vocab , output_dim = 100 , input_length = max_len),
    SimpleRNN(64 , return_sequences = True),
    SimpleRNN(128),
    Dense(vocab , activation = "softmax")

])

In [ ]:
rnn_model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)


In [ ]:
rnn_model.summary()

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim = vocab , output_dim = 100 , input_length = max_len),
    LSTM(64 , return_sequences = True),
    LSTM(128),
    Dense(vocab , activation = "softmax")
])

In [ ]:
lstm_model.compile(
    optimizer = "adam",
    loss = "categorical_crossentropy",
    metrics = ["accuracy"]
)

In [ ]:
lstm_model.summary()

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dense, Dropout


bigru_model = Sequential([

    Embedding(
        input_dim=vocab,
        output_dim=100,
        input_length=max_len
    ),

    Bidirectional(
        GRU(64, return_sequences=True)
    ),

    Bidirectional(
        GRU(128)
    ),

    Dense(vocab, activation="softmax")
])




In [ ]:
bigru_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "bigru_checkpoint.keras",
    save_best_only=False,
    save_weights_only=False
)

history = bigru_model.fit(
    padded_sequences,
    y_one_hot,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    callbacks=[checkpoint]
)

Epoch 1/50
 15/533 ━━━━━━━━━━━━━━━━━━━━ 1:15:14 9s/step - accuracy: 0.0187 - loss: 9.0224

In [ ]:
bigru_model.fit(
    padded_sequences,
    y_one_hot,
    epochs=1,
    batch_size=128,
    validation_split=0.2
)

In [ ]:
len(history.history["loss"])

In [ ]:


bigru_model.save("bigru_model.keras")

In [ ]:
In = []

index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [ ]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
def predictor(model,token,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='post')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]


In [ ]:

seed = "are you a "
generate_text = predictor(bigru_model,token,seed,max_len,10)
print(generate_text)